In [116]:
import pandas_ta as pta
import numpy as np
import pandas as pd
import mplfinance as mpf
from tabulate import tabulate
from utils.KrakenHistoricalData import KrakenHistoricalData
from ggTrader.Signals import Signals
import matplotlib.pyplot as plt
import seaborn as sns


In [117]:
# Ticker
symbols = ["BTC", "ETH"]
interval = "4h"

# Time Range

end = pd.to_datetime("2025-06-30").tz_localize('UTC')
start = end - pd.Timedelta(days=30 * 6)

k = KrakenHistoricalData()

df_multi = k.get_ohlcv_df(symbols, interval=interval)

print(f"\nMultiIndex")
print(df_multi.info())

# select all close
print(df_multi.xs('close', axis=1, level=1).head())

# select only BTC
print(df_multi.xs('BTC', axis=1, level=0).head())

# list of tickers
print(df_multi.columns.levels[0].tolist())


MultiIndex
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 5472 entries, 2023-01-01 00:00:00+00:00 to 2025-06-30 20:00:00+00:00
Freq: 4h
Data columns (total 16 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   (BTC, open)    5472 non-null   float32
 1   (BTC, high)    5472 non-null   float32
 2   (BTC, low)     5472 non-null   float32
 3   (BTC, close)   5472 non-null   float32
 4   (BTC, volume)  5472 non-null   float64
 5   (BTC, trades)  5472 non-null   Int64  
 6   (BTC, base)    5472 non-null   object 
 7   (BTC, quote)   5472 non-null   object 
 8   (ETH, open)    5472 non-null   float32
 9   (ETH, high)    5472 non-null   float32
 10  (ETH, low)     5472 non-null   float32
 11  (ETH, close)   5472 non-null   float32
 12  (ETH, volume)  5472 non-null   float64
 13  (ETH, trades)  5472 non-null   Int64  
 14  (ETH, base)    5472 non-null   object 
 15  (ETH, quote)   5472 non-null   object 
dtypes: Int64(2), float32(8), fl

In [118]:
def process_ohlcv(df):
    data = {}
    col = ['open', 'high', 'low', 'close', 'volume']
    for c in col:
        data[c] = df.xs(c, axis=1, level=1)
    return data


data = process_ohlcv(df_multi)

print("\nProcessed Data")
for col in data.keys():
    print(f"{col}:")
    print(data[col].head())



Processed Data
open:
                                    BTC          ETH
2023-01-01 00:00:00+00:00  16528.699219  1195.000000
2023-01-01 04:00:00+00:00  16519.300781  1194.150024
2023-01-01 08:00:00+00:00  16512.400391  1192.199951
2023-01-01 12:00:00+00:00  16500.199219  1193.430054
2023-01-01 16:00:00+00:00  16550.000000  1197.280029
high:
                                    BTC          ETH
2023-01-01 00:00:00+00:00  16530.000000  1195.000000
2023-01-01 04:00:00+00:00  16542.400391  1195.000000
2023-01-01 08:00:00+00:00  16538.400391  1195.109985
2023-01-01 12:00:00+00:00  16550.000000  1197.719971
2023-01-01 16:00:00+00:00  16573.099609  1197.569946
low:
                                    BTC          ETH
2023-01-01 00:00:00+00:00  16505.199219  1193.000000
2023-01-01 04:00:00+00:00  16506.900391  1190.859985
2023-01-01 08:00:00+00:00  16490.000000  1192.199951
2023-01-01 12:00:00+00:00  16497.699219  1192.699951
2023-01-01 16:00:00+00:00  16531.199219  1193.800049
close:
      

In [119]:
# testing out how I can apply signals to the entire multiindex dataframe
def calc_signals(df: pd.DataFrame, adx_length: int = 14) -> pd.DataFrame:

    df_signals = df.copy()
    if not isinstance(df.columns, pd.MultiIndex):
        print("Not a multiindex dataframe")
        return df_signals

    tickers = df.columns.levels[0].tolist()

    for ticker in tickers:
        df_single = df.xs(ticker, axis=1, level=0)
        adx = pta.adx(df_single['high'],
                      df_single['low'],
                      df_single['close'],
                      length=adx_length)
        adx_cols = adx.columns.tolist()
        new_cols = []
        for col in adx_cols:
            new_cols.append((ticker, col.lower()))
        adx.columns = pd.MultiIndex.from_tuples(new_cols)
        df_signals = pd.concat([df_signals, adx], axis=1)
    return df_signals.sort_index(axis=1,level=0)


signals = calc_signals(df_multi)
print(signals.info())
print(signals['BTC'].tail())

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 5472 entries, 2023-01-01 00:00:00+00:00 to 2025-06-30 20:00:00+00:00
Freq: 4h
Data columns (total 24 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   (BTC, adx_14)     5459 non-null   float64
 1   (BTC, adxr_14_2)  5457 non-null   float64
 2   (BTC, base)       5472 non-null   object 
 3   (BTC, close)      5472 non-null   float32
 4   (BTC, dmn_14)     5459 non-null   float64
 5   (BTC, dmp_14)     5459 non-null   float64
 6   (BTC, high)       5472 non-null   float32
 7   (BTC, low)        5472 non-null   float32
 8   (BTC, open)       5472 non-null   float32
 9   (BTC, quote)      5472 non-null   object 
 10  (BTC, trades)     5472 non-null   Int64  
 11  (BTC, volume)     5472 non-null   float64
 12  (ETH, adx_14)     5459 non-null   float64
 13  (ETH, adxr_14_2)  5457 non-null   float64
 14  (ETH, base)       5472 non-null   object 
 15  (ETH, close)      5472 non-null 